# Chapter 9: Time Series Analysis

## Technical requirements

In [1]:
import polars as pl

In [2]:
lf = pl.scan_csv('../data/toronto_weather.csv')

In [4]:
lf.head().collect()

datetime,temperature,wind_speed,pressure,humidity
str,f64,f64,f64,f64
"""2012-10-01 12:00:00""",null,null,null,null
"""2012-10-01 13:00:00""",286.26,3.0,1012.0,82.0
"""2012-10-01 14:00:00""",286.262541,3.0,1011.0,81.0
"""2012-10-01 15:00:00""",286.269518,3.0,1011.0,79.0
"""2012-10-01 16:00:00""",286.276496,3.0,1010.0,77.0


In [5]:
lf = lf.with_columns(pl.col('temperature')-273.15)

In [6]:
lf.head().collect()

datetime,temperature,wind_speed,pressure,humidity
str,f64,f64,f64,f64
"""2012-10-01 12:00:00""",null,null,null,null
"""2012-10-01 13:00:00""",13.11,3.0,1012.0,82.0
"""2012-10-01 14:00:00""",13.112541,3.0,1011.0,81.0
"""2012-10-01 15:00:00""",13.119518,3.0,1011.0,79.0
"""2012-10-01 16:00:00""",13.126496,3.0,1010.0,77.0


## Working with date and time

### How to do it...

In [23]:
lf_date_parsed = pl.scan_csv('../data/toronto_weather.csv', try_parse_dates=True)
lf_date_parsed.head().collect()

datetime,temperature,wind_speed,pressure,humidity
datetime[μs],f64,f64,f64,f64
2012-10-01 12:00:00,null,null,null,null
2012-10-01 13:00:00,286.26,3.0,1012.0,82.0
2012-10-01 14:00:00,286.262541,3.0,1011.0,81.0
2012-10-01 15:00:00,286.269518,3.0,1011.0,79.0
2012-10-01 16:00:00,286.276496,3.0,1010.0,77.0


In [24]:
lf_date_parsed.collect_schema(), lf_date_parsed.collect_schema().dtypes()

(Schema([('datetime', Datetime(time_unit='us', time_zone=None)),
         ('temperature', Float64),
         ('wind_speed', Float64),
         ('pressure', Float64),
         ('humidity', Float64)]),
 [Datetime(time_unit='us', time_zone=None),
  Float64,
  Float64,
  Float64,
  Float64])

In [7]:
lf = lf.with_columns(
    pl.col('datetime').str.to_datetime()
)
lf.head().collect()

datetime,temperature,wind_speed,pressure,humidity
datetime[μs],f64,f64,f64,f64
2012-10-01 12:00:00,null,null,null,null
2012-10-01 13:00:00,13.11,3.0,1012.0,82.0
2012-10-01 14:00:00,13.112541,3.0,1011.0,81.0
2012-10-01 15:00:00,13.119518,3.0,1011.0,79.0
2012-10-01 16:00:00,13.126496,3.0,1010.0,77.0


In [26]:
(
    lf
    .select(
        'datetime',
        pl.col('datetime').dt.year().alias('year'),
        pl.col('datetime').dt.month().alias('month'),
        pl.col('datetime').dt.day().alias('day'),
        pl.col('datetime').dt.time().alias('time'),
        pl.col('datetime').dt.hour().alias('hour')
    )
    .head().collect()
)

datetime,year,month,day,time,hour
datetime[μs],i32,i8,i8,time,i8
2012-10-01 12:00:00,2012,10,1,12:00:00,12
2012-10-01 13:00:00,2012,10,1,13:00:00,13
2012-10-01 14:00:00,2012,10,1,14:00:00,14
2012-10-01 15:00:00,2012,10,1,15:00:00,15
2012-10-01 16:00:00,2012,10,1,16:00:00,16


In [27]:
from datetime import datetime


filtered_lf = (
    lf
    .filter(
        pl.col('datetime').dt.date().is_between(
            pl.date(2017, 1, 1), pl.date(2017, 12, 31)
        ),
        pl.col('datetime').dt.hour() < 12
    )
)
filtered_lf.head().collect()

datetime,temperature,wind_speed,pressure,humidity
datetime[μs],f64,f64,f64,f64
2017-01-01 00:00:00,2.44,3.0,1001.0,92.0
2017-01-01 01:00:00,2.19,9.0,1003.0,86.0
2017-01-01 02:00:00,2.41,7.0,1003.0,86.0
2017-01-01 03:00:00,2.42,5.0,1003.0,74.0
2017-01-01 04:00:00,1.77,6.0,1006.0,69.0


In [28]:
(
    filtered_lf
    .select(
        pl.col('datetime').dt.year().unique()
        .implode()
        .list.len()
        .alias('year_cnt'),
        pl.col('datetime').dt.hour().unique()
        .implode()
        .list.len()
        .alias('hour_cnt')
    )
    .head()
    .collect()
)

year_cnt,hour_cnt
u32,u32
1,12


In [29]:
import pandas as pd

lf_pd = lf.collect().to_pandas()

In [30]:
dt = lf_pd['datetime']

time_zones_df = pd.DataFrame({
    "datetime": dt,
    "replaced_time_zone_toronto": dt.dt.tz_localize("Asia/Shanghai"),
    "converted_time_zone_toronto": (
        dt.dt.tz_localize("UTC")
        .dt.tz_convert("America/Toronto")
    ),
})

time_zones_df.head()

,datetime,replaced_time_zone_toronto,converted_time_zone_toronto
0,2012-10-01 12:00:00,2012-10-01 12:00:00+08:00,2012-10-01 08:00:00-04:00
1,2012-10-01 13:00:00,2012-10-01 13:00:00+08:00,2012-10-01 09:00:00-04:00
2,2012-10-01 14:00:00,2012-10-01 14:00:00+08:00,2012-10-01 10:00:00-04:00
3,2012-10-01 15:00:00,2012-10-01 15:00:00+08:00,2012-10-01 11:00:00-04:00
4,2012-10-01 16:00:00,2012-10-01 16:00:00+08:00,2012-10-01 12:00:00-04:00


In [31]:
time_zones_lf = (
    lf
    .select(
        'datetime',
        pl.col('datetime').dt.replace_time_zone('America/Toronto')
        .alias('replaced_time_zone_toronto'),
        pl.col('datetime').dt.convert_time_zone('America/Toronto')
        .alias('converted_time_zone_toronto')
    )
)
time_zones_lf.head().collect()

datetime,replaced_time_zone_toronto,converted_time_zone_toronto
datetime[μs],"datetime[μs, America/Toronto]","datetime[μs, America/Toronto]"
2012-10-01 12:00:00,2012-10-01 12:00:00 EDT,2012-10-01 08:00:00 EDT
2012-10-01 13:00:00,2012-10-01 13:00:00 EDT,2012-10-01 09:00:00 EDT
2012-10-01 14:00:00,2012-10-01 14:00:00 EDT,2012-10-01 10:00:00 EDT
2012-10-01 15:00:00,2012-10-01 15:00:00 EDT,2012-10-01 11:00:00 EDT
2012-10-01 16:00:00,2012-10-01 16:00:00 EDT,2012-10-01 12:00:00 EDT


### There is more...

In [40]:
df_pl = lf.select('datetime').collect()
dt_pd = df_pl.to_pandas()['datetime']

In [41]:
%%timeit
(
    df_pl
    .select(
        'datetime',
        (pl.col('datetime')-pl.duration(weeks=5)).alias('minus_5weeks'),
        (pl.col('datetime')+pl.duration(milliseconds=5)).alias('plus_5ms'),   
    )
)

133 μs ± 5.64 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [43]:
%%timeit
pd.DataFrame({
    'datetime': dt_pd,
    'minus_5weeks': dt_pd - pd.Timedelta(weeks=5),
    'plus_5ms': dt_pd + pd.Timedelta(milliseconds=5)
})

1.61 ms ± 102 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [49]:
df_pl.select(
    'datetime',
    pl.col('datetime').dt.offset_by(by='1mo').alias('offset_1mo')
)

datetime,offset_1mo
datetime[μs],datetime[μs]
2012-10-01 12:00:00,2012-11-01 12:00:00
2012-10-01 13:00:00,2012-11-01 13:00:00
2012-10-01 14:00:00,2012-11-01 14:00:00
2012-10-01 15:00:00,2012-11-01 15:00:00
2012-10-01 16:00:00,2012-11-01 16:00:00
…,…
2017-11-29 20:00:00,2017-12-29 20:00:00
2017-11-29 21:00:00,2017-12-29 21:00:00
2017-11-29 22:00:00,2017-12-29 22:00:00


## Applying rolling window calculations

### How to do it...

In [56]:
df = pl.DataFrame({"A": [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]})
result = (
    df.with_row_index("idx")
    .with_columns(pl.col("idx").cast(pl.Int64))
    .rolling(index_column="idx", period="3i")
    .agg(
        pl.col("A").mean().alias("rolling_mean"),
        pl.col("A").sum().alias("rolling_sum"),
    )
)
print(result)

shape: (6, 3)
┌─────┬──────────────┬─────────────┐
│ idx ┆ rolling_mean ┆ rolling_sum │
│ --- ┆ ---          ┆ ---         │
│ i64 ┆ f64          ┆ f64         │
╞═════╪══════════════╪═════════════╡
│ 0   ┆ 1.0          ┆ 1.0         │
│ 1   ┆ 1.5          ┆ 3.0         │
│ 2   ┆ 2.0          ┆ 6.0         │
│ 3   ┆ 3.0          ┆ 9.0         │
│ 4   ┆ 4.0          ┆ 12.0        │
│ 5   ┆ 5.0          ┆ 15.0        │
└─────┴──────────────┴─────────────┘


In [57]:
(
    lf
    .select(
        'datetime',
        'temperature',
        pl.col('temperature').rolling_mean(3).alias('3hr_rollign_avg')
    )
    .head()
    .collect()
)

datetime,temperature,3hr_rollign_avg
datetime[μs],f64,f64
2012-10-01 12:00:00,null,null
2012-10-01 13:00:00,13.11,null
2012-10-01 14:00:00,13.112541,null
2012-10-01 15:00:00,13.119518,13.11402
2012-10-01 16:00:00,13.126496,13.119518


In [8]:
daily_avg_temperature_lf = (
    lf
    .select(
        pl.col('datetime').dt.date().alias('date'),
        'temperature'
    )
    .group_by('date', maintain_order=True)
    .agg(
        pl.col('temperature').mean().alias('daily_avg_temp')
    )
)
daily_avg_temperature_lf.head().collect()

date,daily_avg_temp
date,f64
2012-10-01,13.140854
2012-10-02,14.24739
2012-10-03,14.176875
2012-10-04,15.067917
2012-10-05,16.216458


In [59]:
(
    daily_avg_temperature_lf
    .select(
        'date',
        'daily_avg_temp',
        pl.col('daily_avg_temp').rolling_mean(3).alias('3day_rolling_avg'),
        pl.col('daily_avg_temp').rolling_min(3).alias('3day_rolling_min'),
        pl.col('daily_avg_temp').rolling_max(3).alias('3day_rolling_max')
    )
    .head()
    .collect()
)

date,daily_avg_temp,3day_rolling_avg,3day_rolling_min,3day_rolling_max
date,f64,f64,f64,f64
2012-10-01,13.140854,null,null,null
2012-10-02,14.24739,null,null,null
2012-10-03,14.176875,13.85504,13.140854,14.24739
2012-10-04,15.067917,14.497394,14.176875,15.067917
2012-10-05,16.216458,15.15375,14.176875,16.216458


In [9]:
(
    daily_avg_temperature_lf
    .set_sorted('date')
    .select(
        'date',
        'daily_avg_temp',
        pl.col('daily_avg_temp').rolling_mean(3).alias('3day_rolling_avg'),
        pl.col('daily_avg_temp').rolling_mean(
            window_size=3,
            min_samples=1
        )
        .alias('3day_rolling_avg2'),
        pl.col('daily_avg_temp').mean().rolling(
            index_column='date', 
            period='3d',
            closed='right'
        )
        .alias('3day_rolling_avg3')
    )
    .head(10)
    .collect()
)

date,daily_avg_temp,3day_rolling_avg,3day_rolling_avg2,3day_rolling_avg3
date,f64,f64,f64,f64
2012-10-01,13.140854,null,13.140854,13.140854
2012-10-02,14.24739,null,13.694122,13.694122
2012-10-03,14.176875,13.85504,13.85504,13.85504
2012-10-04,15.067917,14.497394,14.497394,14.497394
2012-10-05,16.216458,15.15375,15.15375,15.15375
2012-10-06,15.725417,15.669931,15.669931,15.669931
2012-10-07,10.197083,14.046319,14.046319,14.046319
2012-10-08,6.79625,10.90625,10.90625,10.90625
2012-10-09,6.735,7.909444,7.909444,7.909444


In [66]:
(
    daily_avg_temperature_lf
    .set_sorted('date')
    .rolling(
        'date',
        period='3d'
    )
    .agg(
        pl.col('daily_avg_temp'),
        pl.col('daily_avg_temp').mean().alias('3day_rolling_avg'),
        pl.col('daily_avg_temp').min().alias('3day_rolling_min'),
        pl.col('daily_avg_temp').max().alias('3day_rolling_max'),
    )
    .head(10)
    .collect()
)

date,daily_avg_temp,3day_rolling_avg,3day_rolling_min,3day_rolling_max
date,list[f64],f64,f64,f64
2012-10-01,[13.140854],13.140854,13.140854,13.140854
2012-10-02,"[13.140854, 14.24739]",13.694122,13.140854,14.24739
2012-10-03,"[13.140854, 14.24739, 14.176875]",13.85504,13.140854,14.24739
2012-10-04,"[14.24739, 14.176875, 15.067917]",14.497394,14.176875,15.067917
2012-10-05,"[14.176875, 15.067917, 16.216458]",15.15375,14.176875,16.216458
2012-10-06,"[15.067917, 16.216458, 15.725417]",15.669931,15.067917,16.216458
2012-10-07,"[16.216458, 15.725417, 10.197083]",14.046319,10.197083,16.216458
2012-10-08,"[15.725417, 10.197083, 6.79625]",10.90625,6.79625,15.725417
2012-10-09,"[10.197083, 6.79625, 6.735]",7.909444,6.735,10.197083


In [13]:
import altair as alt
(
    daily_avg_temperature_lf
    .select(
        'date',
        'daily_avg_temp',
        pl.col('daily_avg_temp').rolling_mean(60).alias('60day_rolling_avg')
    )
    .collect()
    .unpivot(
        index='date',
        on=['daily_avg_temp', '60day_rolling_avg'],
        variable_name='series',
        value_name='temperature',
    )
    .plot.line(
        x='date:T',
        y='temperature:Q',
        color=alt.Color(
            'series:N',
            scale=alt.Scale(
                domain=['daily_avg_temp', '60day_rolling_avg'],
                range=['skyblue', 'gray'],
            ),
            legend=alt.Legend(orient='bottom'),
        ),
    )
    .properties(width=800, height=400).interactive()
)

alt.Chart(...)

### There is more...

In [14]:
def get_range(nums):
    range = max(nums) - min(nums)
    return range

(
    daily_avg_temperature_lf
    .with_columns(
        pl.col('daily_avg_temp').rolling_map(get_range, window_size=3).alias('3day_rolling_range')
    )
    .head()
    .collect()
)    


date,daily_avg_temp,3day_rolling_range
date,f64,f64
2012-10-01,13.140854,null
2012-10-02,14.24739,null
2012-10-03,14.176875,1.106536
2012-10-04,15.067917,0.891042
2012-10-05,16.216458,2.039583


In [ ]:
(
    daily_avg_temperature_lf
    .with_columns(
        pl.col('daily_avg_temp').rolling_map(get_range, window_size=3).alias('3day_rolling_range')
    )
    .collect()
    .plot.line(
        x='date', 
        y=['daily_avg_temp', '3day_rolling_range'],
        color=['skyblue', 'gray'],
        width=800,
        height=400
    )
    .opts(legend_position='bottom_right')
)    

:NdOverlay   [Variable]
   :Curve   [date]   (value)

## Resampling techniques

### How to do it...

In [26]:
(
    lf
    .set_sorted('datetime')
    .group_by_dynamic(
        'datetime', every='1w', period='3w', include_boundaries=True, label='left'
    )
    .agg(pl.col('humidity').mean().round(1))
    .head(10)
    .collect()
)

_lower_boundary,_upper_boundary,datetime,humidity
datetime[μs],datetime[μs],datetime[μs],f64
2012-10-01 00:00:00,2012-10-22 00:00:00,2012-10-01 00:00:00,67.2
2012-10-08 00:00:00,2012-10-29 00:00:00,2012-10-08 00:00:00,69.4
2012-10-15 00:00:00,2012-11-05 00:00:00,2012-10-15 00:00:00,75.3
2012-10-22 00:00:00,2012-11-12 00:00:00,2012-10-22 00:00:00,72.8
2012-10-29 00:00:00,2012-11-19 00:00:00,2012-10-29 00:00:00,72.2
2012-11-05 00:00:00,2012-11-26 00:00:00,2012-11-05 00:00:00,72.7
2012-11-12 00:00:00,2012-12-03 00:00:00,2012-11-12 00:00:00,72.9
2012-11-19 00:00:00,2012-12-10 00:00:00,2012-11-19 00:00:00,77.7
2012-11-26 00:00:00,2012-12-17 00:00:00,2012-11-26 00:00:00,75.8


In [30]:
from datetime import datetime

(
    lf
    .set_sorted('datetime')
    .group_by_dynamic(
        'datetime', every='1w'
    )
    .agg(pl.col('humidity').mean().round(1))
    .filter(
        pl.col('datetime').dt.date().is_between(
            datetime(2012,10,1), 
            datetime(2012,10,31)
        )
    )
    .collect()
    .plot.line(
        x='datetime', 
        y='humidity',
        color=['skyblue'],
        width=1000,
        height=400
    )
)


TypeError: _EncodingMixin.encode() got an unexpected keyword argument 'width'

In [27]:
upsampled_df = (
    lf
    .set_sorted('datetime')
    .collect()
    .upsample(
        time_column='datetime', 
        every='30m', 
        maintain_order=True
    )
    .select(
        'datetime', 
        pl.col('humidity')
    )
)
upsampled_df.head(10)

datetime,humidity
datetime[μs],f64
2012-10-01 12:00:00,null
2012-10-01 12:30:00,null
2012-10-01 13:00:00,82.0
2012-10-01 13:30:00,null
2012-10-01 14:00:00,81.0
2012-10-01 14:30:00,null
2012-10-01 15:00:00,79.0
2012-10-01 15:30:00,null
2012-10-01 16:00:00,77.0


In [28]:
(
    upsampled_df
    .with_columns(
        pl.col('humidity').interpolate()
    )
    .head(10)
)

datetime,humidity
datetime[μs],f64
2012-10-01 12:00:00,null
2012-10-01 12:30:00,null
2012-10-01 13:00:00,82.0
2012-10-01 13:30:00,81.5
2012-10-01 14:00:00,81.0
2012-10-01 14:30:00,80.0
2012-10-01 15:00:00,79.0
2012-10-01 15:30:00,78.0
2012-10-01 16:00:00,77.0


In [31]:
(
    upsampled_df
    .with_columns(
        pl.col('humidity').interpolate()
    )
    .filter(
        pl.col('datetime').dt.date().is_between(
            datetime(2012,10,1),
            datetime(2012,10,31)
        )
    )
    .plot.area(
        x='datetime', 
        y='humidity',
        color=['skyblue'],
        width=1000,
        height=400,
        alpha=0.5
    )
)


TypeError: _EncodingMixin.encode() got an unexpected keyword argument 'width'

In [32]:
datetime_with_gaps_lf = (
    lf.filter(
        ~pl.col('datetime').dt.hour().is_in([13,15,16,19])
    )
)

(
    datetime_with_gaps_lf
    .set_sorted('datetime')
    .collect()
    .upsample(
        time_column='datetime', 
        every='1h', 
        maintain_order=True
    )
    .select(
        'datetime', 
        pl.col('humidity')
    )
    .head(10)
)

datetime,humidity
datetime[μs],f64
2012-10-01 12:00:00,null
2012-10-01 13:00:00,null
2012-10-01 14:00:00,81.0
2012-10-01 15:00:00,null
2012-10-01 16:00:00,null
2012-10-01 17:00:00,76.0
2012-10-01 18:00:00,74.0
2012-10-01 19:00:00,null
2012-10-01 20:00:00,70.0


In [ ]:
datetime_range_lf = pl.LazyFrame({
    'datetime': pl.datetime_range(
        start=lf.select(pl.col('datetime').min()).collect()[0,0], 
        end=lf.select(pl.col('datetime').max()).collect()[0,0], 
        interval='30m', 
        eager=True
    )
})

(
    datetime_range_lf
    .join(lf, on='datetime', how='left')
    .select(
        'datetime', 
        pl.col('humidity')
    )
    .collect()
    .head(10)
)


datetime,humidity
datetime[μs],f64
2012-10-01 12:00:00,null
2012-10-01 12:30:00,null
2012-10-01 13:00:00,82.0
2012-10-01 13:30:00,null
2012-10-01 14:00:00,81.0
2012-10-01 14:30:00,null
2012-10-01 15:00:00,79.0
2012-10-01 15:30:00,null
2012-10-01 16:00:00,77.0


## Time series forecasting with the functime library

### Getting ready

In [33]:
import polars as pl

In [34]:
lf = pl.scan_csv('../data/historical_temperatures.csv', try_parse_dates=True)

In [35]:
lf.head().collect()

datetime,city,temperature
datetime[μs],str,f64
2012-10-01 12:00:00,"""Toronto""",null
2012-10-01 13:00:00,"""Toronto""",286.26
2012-10-01 14:00:00,"""Toronto""",286.262541
2012-10-01 15:00:00,"""Toronto""",286.269518
2012-10-01 16:00:00,"""Toronto""",286.276496


In [36]:
lf.select('city').unique().sort('city').collect()

city
str
"""Las Vegas"""
"""New York"""
"""San Francisco"""
"""Seattle"""
"""Toronto"""
"""Vancouver"""


In [44]:
lf.group_by('city', maintain_order=True).head(3).collect()

city,datetime,temperature
str,datetime[μs],f64
"""Toronto""",2012-10-01 12:00:00,null
"""Toronto""",2012-10-01 13:00:00,286.26
"""Toronto""",2012-10-01 14:00:00,286.262541
"""Vancouver""",2012-10-01 12:00:00,null
"""Vancouver""",2012-10-01 13:00:00,284.63
…,…,…
"""New York""",2012-10-01 13:00:00,288.22
"""New York""",2012-10-01 14:00:00,288.247676
"""Las Vegas""",2012-10-01 12:00:00,null


In [49]:
lf.select(pl.all().head(3).over('city', mapping_strategy='explode')).collect()

datetime,city,temperature
datetime[μs],str,f64
2012-10-01 12:00:00,"""Toronto""",null
2012-10-01 13:00:00,"""Toronto""",286.26
2012-10-01 14:00:00,"""Toronto""",286.262541
2012-10-01 12:00:00,"""Vancouver""",null
2012-10-01 13:00:00,"""Vancouver""",284.63
…,…,…
2012-10-01 13:00:00,"""New York""",288.22
2012-10-01 14:00:00,"""New York""",288.247676
2012-10-01 12:00:00,"""Las Vegas""",null


In [43]:
lf.group_by('city', maintain_order=True).agg(pl.all().head(3)).explode('datetime', 'temperature').collect()

city,datetime,temperature
str,datetime[μs],f64
"""Toronto""",2012-10-01 12:00:00,null
"""Toronto""",2012-10-01 13:00:00,286.26
"""Toronto""",2012-10-01 14:00:00,286.262541
"""Vancouver""",2012-10-01 12:00:00,null
"""Vancouver""",2012-10-01 13:00:00,284.63
…,…,…
"""New York""",2012-10-01 13:00:00,288.22
"""New York""",2012-10-01 14:00:00,288.247676
"""Las Vegas""",2012-10-01 12:00:00,null


### How to do it...

In [50]:
time_col, entity_col, value_col = lf.collect_schema().names()

y = (
    lf
    .group_by_dynamic(
        time_col,
        every='1mo',
        group_by=entity_col,
    )
    .agg(
        (pl.col('temperature').mean()-273.15).round(1),
    )
)

In [51]:
y.group_by('city').head(3).collect()

city,datetime,temperature
str,datetime[μs],f64
"""Vancouver""",2012-10-01 00:00:00,10.1
"""Vancouver""",2012-11-01 00:00:00,7.2
"""Vancouver""",2012-12-01 00:00:00,4.4
"""Toronto""",2012-10-01 00:00:00,10.3
"""Toronto""",2012-11-01 00:00:00,4.3
…,…,…
"""San Francisco""",2012-11-01 00:00:00,13.4
"""San Francisco""",2012-12-01 00:00:00,10.3
"""Las Vegas""",2012-10-01 00:00:00,20.9


In [54]:
def create_train_test_sets(
    y, 
    entity_col, 
    time_col, 
    test_size
):
    from functime.cross_validation import train_test_split

    X = y.select(entity_col, time_col)
    y_train, y_test = (
        y
        .select(entity_col, time_col, value_col)
        .pipe(train_test_split(test_size))
    )
    X_train, X_test = X.pipe(train_test_split(test_size))

    return X_train, X_test, y_train, y_test

In [55]:
test_size = 3
X_train, X_test, y_train, y_test = create_train_test_sets(y, entity_col, time_col, test_size)

In [58]:
def predict_with_linear_model(
    lags, 
    freq, 
    y_train,
    fh
):
    from functime.forecasting import linear_model

    forecaster = linear_model(lags=lags, freq=freq)
    forecaster.fit(y=y_train)
    y_pred = forecaster.predict(fh=fh)
    return y_pred

In [59]:
y_pred = predict_with_linear_model(24, '1mo', y_train, test_size)

d:\Programs\anaconda3\envs\py314\Lib\site-packages\flaml\__init__.py:20: UserWarning: flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
  warnings.warn("flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.")


In [ ]:
from functime.metrics import mase
scores = mase(y_true=y_test, y_pred=y_pred, y_train=y_train)

In [ ]:
display(y_pred, scores)

city,datetime,temperature
str,datetime[μs],f64
"""San Francisco""",2017-09-01 00:00:00,19.970327
"""San Francisco""",2017-10-01 00:00:00,18.315983
"""San Francisco""",2017-11-01 00:00:00,16.368908
"""Vancouver""",2017-09-01 00:00:00,16.471989
"""Vancouver""",2017-10-01 00:00:00,14.267763
"""Vancouver""",2017-11-01 00:00:00,11.530046
"""Las Vegas""",2017-09-01 00:00:00,33.356968
"""Las Vegas""",2017-10-01 00:00:00,26.222851
"""Las Vegas""",2017-11-01 00:00:00,19.330387


city,mase
str,f64
"""Toronto""",1.341055
"""Seattle""",1.286418
"""San Francisco""",0.542017
"""Las Vegas""",1.151459
"""New York""",1.082172
"""Vancouver""",0.737569


In [ ]:
actual_viz = (
    y
    .collect()
    .plot.line(
        x='datetime', y='temperature', by='city', subplots=True

    )
    .cols(2)
) 

pred_viz = (
    y_pred
    .plot.line(
        x='datetime', y='temperature', by='city', subplots=True
    )
    .cols(2)
)

actual_viz * pred_viz


:NdLayout   [city]
   :Overlay
      .Curve.I  :Curve   [datetime]   (temperature)
      .Curve.II :Curve   [datetime]   (temperature)

### There is more...

In [ ]:
from functime.seasonality import add_calendar_effects

y_features = (
    lf
    .group_by_dynamic(
        time_col,
        every='1mo',
        group_by=entity_col,
    )
    .agg(
        (pl.col('temperature').mean()-273.15).round(1),
        pl.col(value_col).ts.binned_entropy(bin_count=10)
        .alias('binned_entropy'),
        pl.col(value_col).ts.lempel_ziv_complexity(threshold=3)
        .alias('lempel_ziv_complexity'),
        pl.col(value_col).ts.longest_streak_above_mean()
        .alias('longest_streak_above_mean')
    )
    .pipe(add_calendar_effects(['month']))
)
y_features.head().collect()

city,datetime,temperature,binned_entropy,lempel_ziv_complexity,longest_streak_above_mean,month
str,datetime[μs],f64,f64,f64,u64,cat
"""Toronto""",2012-10-01 00:00:00,10.3,2.02934,0.051913,94,"""10"""
"""Toronto""",2012-11-01 00:00:00,4.3,1.969688,0.051389,87,"""11"""
"""Toronto""",2012-12-01 00:00:00,1.1,2.10535,0.051075,87,"""12"""
"""Toronto""",2013-01-01 00:00:00,-2.1,2.082969,0.051075,171,"""1"""
"""Toronto""",2013-02-01 00:00:00,-3.4,1.946257,0.053571,130,"""2"""
